# 03_Fingerprint Feature Extraction

## Objective
The objective of this notebook is to extract fingerprint features and create a model-ready dataset.

## Method
Feature extraction from preprocessed fingerprint images.

## Input
Processed fingerprint images.

## Processing Tasks

1. Load and Validate Fingerprint Images.
2. HOG Feature Extraction.
3. LBP Feature Extraction.
4. Combine HOG and LBP Features.
5. Save Combined Fingerprint Features.

## Output
- Fingerprint feature vectors.
- Labeled fingerprint feature dataset.
- Exported features for machine learning.

## Purpose
Produce reliable fingerprint representations for recognition model training.


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.feature import hog, local_binary_pattern

# Project Paths
PROJECT_PATH = Path(r"E:\Multimodal Biometric Verification System")

INPUT_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Fingerprint"
    / "Fingerprint_CLAHE"
)

OUTPUT_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Fingerprint"
    / "Fingerprint_Features"
)

MODEL_PATH = PROJECT_PATH / "Models" / "Fingerprint"

# Selected Users
USERS = ["User_01", "User_02"]

# HOG Configuration
HOG_ORIENTATIONS = 9
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)
HOG_BLOCK_NORM = "L2-Hys"

# LBP Configuration
LBP_POINTS = 8
LBP_RADIUS = 1
LBP_METHOD = "uniform"

# Create Output Directories
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
MODEL_PATH.mkdir(parents=True, exist_ok=True)

for user_id in USERS:
    (OUTPUT_PATH / user_id).mkdir(parents=True, exist_ok=True)

print("Fingerprint Feature Extraction Environment Configured")
print("---------------------------------------------")
print(f"Input Path: {INPUT_PATH}")
print(f"Output Path: {OUTPUT_PATH}")
print(f"Model Path: {MODEL_PATH}")
print(f"Selected Users: {USERS}")
print(f"HOG Orientations: {HOG_ORIENTATIONS}")
print(f"LBP Points: {LBP_POINTS}")
print(f"LBP Radius: {LBP_RADIUS}")

### 1. Load and Validate Fingerprint Images


In [ ]:
fingerprint_images = {}
validation_records = []
unreadable_images = []

for user_id in USERS:
    user_path = INPUT_PATH / user_id
    image_files = sorted(user_path.glob("*.BMP"))

    for image_file in image_files:
        image = cv2.imread(str(image_file), cv2.IMREAD_GRAYSCALE)

        if image is None:
            unreadable_images.append(str(image_file))
            continue

        fingerprint_images[(user_id, image_file.name)] = image

        validation_records.append({
            "User_ID": user_id,
            "Image_Name": image_file.name,
            "Height": image.shape[0],
            "Width": image.shape[1],
            "Data_Type": str(image.dtype),
            "Min_Pixel_Value": int(image.min()),
            "Max_Pixel_Value": int(image.max())
        })

fingerprint_validation_df = pd.DataFrame(validation_records)

print("Fingerprint Images Loaded Successfully")
print("---------------------------------------------")
print(f"Total Images Loaded: {len(fingerprint_images)}")

print("\nImages per User:")
print(fingerprint_validation_df["User_ID"].value_counts().sort_index())

print(f"\nUnreadable Images: {len(unreadable_images)}")

print("\nImage Dimensions:")
print(
    fingerprint_validation_df[["Height", "Width"]]
    .value_counts()
    .sort_index()
)

print("\nData Types:")
print(fingerprint_validation_df["Data_Type"].value_counts())

print("\nUnreadable Files:")
print(unreadable_images if unreadable_images else "None")

print("\nValidation Preview:")
display(fingerprint_validation_df.head())

### 2. HOG Feature Extraction


In [ ]:
hog_features = []
hog_records = []

for (user_id, image_name), image in fingerprint_images.items():

    features = hog(
        image,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm=HOG_BLOCK_NORM,
        visualize=False,
        feature_vector=True
    )

    hog_features.append(features)

    record = {
        "User_ID": user_id,
        "Image_Name": image_name
    }

    for index, value in enumerate(features):
        record[f"HOG_{index}"] = value

    hog_records.append(record)

hog_df = pd.DataFrame(hog_records)

print("HOG Feature Extraction Completed")
print("---------------------------------------------")
print(f"Total Images Processed: {len(hog_df)}")
print(f"Number of HOG Features per Image: {len(hog_features[0])}")
print(f"HOG Dataset Shape: {hog_df.shape}")

print("\nImages per User:")
print(hog_df["User_ID"].value_counts().sort_index())

print("\nMissing Values:")
print(hog_df.isnull().sum().sum())

print("\nHOG Features Preview:")
display(hog_df.iloc[:, :12].head())

### 3. LBP Feature Extraction


In [ ]:
lbp_records = []

LBP_HISTOGRAM_BINS = LBP_POINTS + 2

for (user_id, image_name), image in fingerprint_images.items():

    lbp_image = local_binary_pattern(
        image,
        P=LBP_POINTS,
        R=LBP_RADIUS,
        method=LBP_METHOD
    )

    lbp_histogram, _ = np.histogram(
        lbp_image.ravel(),
        bins=np.arange(0, LBP_HISTOGRAM_BINS + 1),
        range=(0, LBP_HISTOGRAM_BINS)
    )

    lbp_histogram = lbp_histogram.astype(np.float64)
    lbp_histogram /= lbp_histogram.sum()

    record = {
        "User_ID": user_id,
        "Image_Name": image_name
    }

    for index, value in enumerate(lbp_histogram):
        record[f"LBP_{index}"] = value

    lbp_records.append(record)

lbp_df = pd.DataFrame(lbp_records)

print("LBP Feature Extraction Completed")
print("---------------------------------------------")
print(f"Total Images Processed: {len(lbp_df)}")
print(f"Number of LBP Features per Image: {LBP_HISTOGRAM_BINS}")
print(f"LBP Dataset Shape: {lbp_df.shape}")

print("\nImages per User:")
print(lbp_df["User_ID"].value_counts().sort_index())

print("\nMissing Values:")
print(lbp_df.isnull().sum().sum())

print("\nLBP Histogram Sum (First 5 Images):")
print(lbp_df.iloc[:, 2:].sum(axis=1).head())

print("\nLBP Features Preview:")
display(lbp_df.head())

### 4. Combine HOG and LBP Features


In [ ]:
fingerprint_features_df = pd.merge(
    lbp_df,
    hog_df,
    on=["User_ID", "Image_Name"],
    how="inner",
    validate="one_to_one"
)

# Identify Feature Columns
lbp_columns = [
    column for column in fingerprint_features_df.columns
    if column.startswith("LBP_")
]

hog_columns = [
    column for column in fingerprint_features_df.columns
    if column.startswith("HOG_")
]

feature_columns = lbp_columns + hog_columns

print("Fingerprint Feature Combination Completed")
print("---------------------------------------------")
print(f"Total Images: {len(fingerprint_features_df)}")
print(f"LBP Features: {len(lbp_columns)}")
print(f"HOG Features: {len(hog_columns)}")
print(f"Total Numerical Features: {len(feature_columns)}")
print(f"Combined Dataset Shape: {fingerprint_features_df.shape}")

print("\nImages per User:")
print(fingerprint_features_df["User_ID"].value_counts().sort_index())

print("\nMissing Values:")
print(fingerprint_features_df.isnull().sum().sum())

print("\nDuplicate Image Records:")
print(
    fingerprint_features_df.duplicated(
        subset=["User_ID", "Image_Name"]
    ).sum()
)

print("\nCombined Features Preview:")
display(fingerprint_features_df.iloc[:, :15].head())

### 5. Save Combined Fingerprint Features


In [ ]:
# Create Output Directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Save Combined Features
FEATURES_FILE = OUTPUT_PATH / "Fingerprint_Combined_Features.csv"

fingerprint_features_df.to_csv(
    FEATURES_FILE,
    index=False
)

print("Fingerprint Features Saved Successfully")
print("---------------------------------------------")
print(f"File Path: {FEATURES_FILE}")
print(f"Dataset Shape: {fingerprint_features_df.shape}")
print(f"Total Images: {len(fingerprint_features_df)}")
print(f"Total Numerical Features: {len(feature_columns)}")
print(f"File Exists: {FEATURES_FILE.exists()}")